In [1]:
!pip install chromadb sentence-transformers

In [2]:
import os

BASE_DIR = os.path.abspath(".")
PERSIST_DIR = os.path.join(BASE_DIR, "vector_db", "chroma")

os.makedirs(PERSIST_DIR, exist_ok=True)

print("Persist directory:", PERSIST_DIR)
print("Exists?", os.path.exists(PERSIST_DIR))


Persist directory: C:\Users\ASUS\jan\chroma_processing\NEW DB for 100\vector_db\chroma
Exists? True


In [3]:
import json
from pathlib import Path

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

In [4]:
DATA_FILE = Path("data's.json")

with open(DATA_FILE, "r", encoding="utf-8") as f:
    herbs = json.load(f)

print("Herbs loaded:", len(herbs))

Herbs loaded: 101


In [5]:
def normalize_metadata(meta):
    clean = {}
    for k, v in meta.items():
        if isinstance(v, list):
            clean[k] = ", ".join(map(str, v))
        else:
            clean[k] = str(v)
    return clean

In [6]:
ids = []
documents = []
metadatas = []

for idx, herb in enumerate(herbs):
    ids.append(f"HERB_{idx+1:04d}")
    documents.append(herb["document"])
    metadatas.append(normalize_metadata(herb["metadata"]))

print(len(ids), len(documents), len(metadatas))

101 101 101


In [7]:
class SBERTEmbeddingFunction:
    def __init__(self, model_name="paraphrase-multilingual-MiniLM-L12-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def __call__(self, input):
        return self.model.encode(input).tolist()

    def name(self):
        return f"sbert-{self.model_name}"

       

In [8]:
chroma_client = chromadb.Client(
    Settings(
        persist_directory=PERSIST_DIR,
        anonymized_telemetry=False
    )
)

In [9]:
embedding_function = SBERTEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="herbal_knowledge_base",
    embedding_function=embedding_function
)

print("Collection ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection ready


In [10]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Total vectors:", collection.count())


Total vectors: 101


In [11]:
print("vector_db exists:", os.path.exists("vector_db"))
print("chroma exists:", os.path.exists(PERSIST_DIR))
print("files:", os.listdir(PERSIST_DIR))


vector_db exists: True
chroma exists: True
files: []


In [12]:
print("Total herbs in vector DB:", collection.count())


Total herbs in vector DB: 101


In [25]:
query = "What are the traditional medicinal uses of Chrysanthemum indicum?"


query_embedding = embedding_function.model.encode([query]).tolist()


results = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)


for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:\n", doc[:400], "...")



Result 1:
 Chrysanthemum indicum, commonly known as Indian chrysanthemum, wild chrysanthemum, or Guldaudi in India, belongs to the family Asteraceae and is a perennial aromatic herb characterized by its erect branching stems, lobed or pinnatifid leaves, and distinctive yellow to white composite flower heads that exhibit a strong fragrance and bioactive profile. Native to East Asia including China, Japan, and ...
